In [ ]:
%pip install pandas
%pip install matplotlib
%pip install seaborn
%pip install statsmodels

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='urllib3')
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df_new = pd.read_json('./Task_2_new/train_data.json')

In [ ]:
flat_labels = df_new['labels'].explode()
label_percentages = flat_labels.value_counts(normalize=True)

new_labels = [
    'None',
    'GENERATION_FAILURE',
    'LEAKED_INSTRUCTIONS',
    'UNGROUNDED_INJECTION',
    'REPETITIVE_CONTENT',
    'GROUNDED_OVERGENERATION'
]

none_percentage = label_percentages.get(new_labels[0], 0)
generation_failure_percentage = label_percentages.get(new_labels[1], 0)
leaked_instructions_percentage = label_percentages.get(new_labels[2], 0)
ungrounded_injection_percentage = label_percentages.get(new_labels[3], 0)
repetitive_content_percentage = label_percentages.get(new_labels[4], 0)
grounded_overgeneration_percentage = label_percentages.get(new_labels[5], 0)

print(
    f"Percentage of non-spurious entries: {none_percentage:.2%}"
)
print(
    f"Percentage of label 'GENERATION_FAILURE': {generation_failure_percentage:.2%}"
)
print(
    f"Percentage of label 'LEAKED_INSTRUCTIONS': {leaked_instructions_percentage:.2%}"
)
print(
    f"Percentage of label 'UNGROUNDED_INJECTION': {ungrounded_injection_percentage:.2%}"
)
print(
    f"Percentage of label 'REPETITIVE_CONTENT': {repetitive_content_percentage:.2%}"
)
print(
    f"Percentage of label 'GROUNDED_OVERGENERATION': {grounded_overgeneration_percentage:.2%}"
)

In [ ]:
df_new_expanded = df_new.explode(['sentences', 'labels'])

df_new_expanded['word_count'] = df_new_expanded['sentences'].str.findall(r'[\w-]+').str.len()
none_mask = df_new_expanded['labels'] == new_labels[0]

print(
    f'{df_new_expanded.groupby(none_mask)['word_count'].mean()}'
)

In [ ]:
generation_failure_mask = df_new_expanded['labels'] == new_labels[1]
leaked_instructions_mask = df_new_expanded['labels'] == new_labels[2]
ungrounded_injection_mask = df_new_expanded['labels'] == new_labels[3]
repetitive_content_mask = df_new_expanded['labels'] == new_labels[4]
grounded_overgeneration_mask = df_new_expanded['labels'] == new_labels[5]

print(
    f"Average sentence length of label 'GENERATION_FAILURE': {df_new_expanded[generation_failure_mask]['word_count'].mean()}"
)
print(
    f"Average sentence length of label 'LEAKED_INSTRUCTIONS': {df_new_expanded[leaked_instructions_mask]['word_count'].mean()}"
)
print(
    f"Average sentence length of label 'UNGROUNDED_INJECTION': {df_new_expanded[ungrounded_injection_mask]['word_count'].mean()}"
)
print(
    f"Average sentence length of label 'REPETITIVE_CONTENT': {df_new_expanded[repetitive_content_mask]['word_count'].mean()}"
)
print(
    f"Average sentence length of label 'GROUNDED_OVERGENERATION': {df_new_expanded[grounded_overgeneration_mask]['word_count'].mean()}"
)

In [ ]:
groups = pd.Series([none_percentage, leaked_instructions_percentage, ungrounded_injection_percentage, repetitive_content_percentage, grounded_overgeneration_percentage], index=['None', 'LEAKED_INSTRUCTIONS', 'UNGROUNDED_INJECTION', 'REPETITIVE_CONTENT', 'GROUNDED_OVERGENERATION'], name='groups')
groups.plot.pie(figsize=(6, 6), title='Label groups')

In [ ]:
new_bins = range(0, df_new_expanded['word_count'].max() + 10, 10)
w_o_none = df_new_expanded[df_new_expanded['labels'] != 'None']

df_new_expanded['groups'] = pd.cut(df_new_expanded['word_count'], bins=new_bins, right=False)
new_plot_data = df_new_expanded.groupby(['groups', 'labels']).size().unstack().fillna(0)
new_plot_data.plot.bar(figsize=(12, 6))

df_new_expanded['has_label'] = (
    df_new_expanded['labels'] != 'None'
)
new_grouped_plot_data = (
    df_new_expanded
    .groupby(['groups', 'has_label'])
    .size()
    .unstack(fill_value=0)
)
new_grouped_plot_data.plot.bar(
    figsize=(12, 6),
)
plt.xlabel('Word Count Group')
plt.ylabel('Count')
plt.show()

w_o_none['groups'] = pd.cut(w_o_none['word_count'], bins=new_bins, right=False)
w_o_plot_data = w_o_none.groupby(['groups', 'labels']).size().unstack().fillna(0)
w_o_plot_data.plot.bar(figsize=(12, 6))

In [ ]:
new_plot_data_2 = (
    df_new_expanded
    .groupby(['word_count', 'labels'])
    .size()
    .unstack(fill_value=0)
)

new_plot_data_norm = new_plot_data_2.div(
    new_plot_data_2.sum(axis=1),
    axis=0
)

plt.figure(figsize=(12,6))
sns.heatmap(
    new_plot_data_norm,
    cmap='viridis'
)
plt.xlabel('Label')
plt.ylabel('Word Count')
plt.show()


new_plot_data_3 = (
    df_new_expanded
    .groupby(['groups', 'labels'])
    .size()
    .unstack(fill_value=0)
)

new_plot_data_norm_2 = new_plot_data_3.div(
    new_plot_data_3.sum(axis=1),
    axis=0
)

plt.figure(figsize=(12,6))
sns.heatmap(
    new_plot_data_norm_2,
    cmap='viridis'
)
plt.xlabel('Label')
plt.ylabel('Word Count')
plt.show()


new_counts = (
    df_new_expanded
    .groupby(['word_count', 'labels'])
    .size()
    .reset_index(name='count')
)
new_counts['prob'] = new_counts.groupby('word_count')['count'].transform(lambda x: x / x.sum())

plt.figure(figsize=(12,6))
ax_4 = sns.lineplot(
    data=new_counts,
    x='word_count',
    y='prob',
    hue='labels'
)
plt.xlabel('Word Count')
plt.ylabel('Probability')
sns.move_legend(ax_4, "upper left", bbox_to_anchor=(1, 1))
plt.show()

plt.figure(figsize=(12,6))

ax_5 = sns.scatterplot(
    data=new_counts,
    x='word_count',
    y='prob',
    hue='labels',
    alpha=0.6
)
plt.xlabel('Word Count')
plt.ylabel('Probability')
sns.move_legend(ax_5, "upper left", bbox_to_anchor=(1, 1))
plt.show()


plt.figure(figsize=(12,6))

ax_6 = sns.scatterplot(
    data=new_counts,
    x='word_count',
    y='prob',
    hue='labels',
    alpha=0.6
)

for label, group in new_counts.groupby('labels'):
    sns.regplot(
        data=group,
        x='word_count',
        y='prob',
        lowess=True,
        scatter=False,
        ax=ax_6,
        label=f"{label} trend"
    )

plt.xlabel('Word Count')
plt.ylabel('Probability')
sns.move_legend(ax_6, "upper left", bbox_to_anchor=(1, 1))
plt.show()